# Revisit HDF5 and extract temperatures

<!-- index:description -->
Explore the structure of OpenMC neutron HDF5 files, inspect groups and attributes, extract MT temperatures, and compare temperature availability across nuclear-data libraries.
<!-- /index:description -->

<!-- generated:structure -->
```text
Revisit HDF5 and extract temperatures
├── 1. Inspect HDF5 structure
├── 2. Inspect attributes
└── 3. Get metadata
```
<!-- /generated:structure -->


In [1]:
import h5py
import pandas as pd

from openmc_workbench_config import DATA_ROOT

df = pd.read_feather('0_2_cataloging_nuclear_data_libraries_df.feather')
df

,materials,path,library,MT
0,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,2
1,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,102
2,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,204
3,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,301
4,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,444
...,...,...,...,...
219209,Fm257,jendl-5/neutron/Fm257.h5,jendl-5,750
219210,Fm257,jendl-5/neutron/Fm257.h5,jendl-5,799
219211,Fm257,jendl-5/neutron/Fm257.h5,jendl-5,800
219212,Fm257,jendl-5/neutron/Fm257.h5,jendl-5,849


## 1. Inspect HDF5 structure

In [2]:
def get_nameobj(filename):
    with h5py.File(DATA_ROOT / filename) as ff:
        nuclide = next(iter(ff.values()))

        return {
            (name, type(obj).__name__)
            for name, obj in nuclide.items()
        }

name_obj = set()
for filename in df['path'].unique():
    name_obj.update(get_nameobj(filename))
name_obj

{('energy', 'Group'),
 ('fission_energy_release', 'Group'),
 ('kTs', 'Group'),
 ('reactions', 'Group'),
 ('total_nu', 'Group'),
 ('urr', 'Group')}

| Group name | Definition |
|---|---|
| `kTs` | Temperatures expressed as $k_BT$, in eV |
| `energy` | Incident neutron energy grids at different temperatures |
| `reactions` | Nuclear reaction data, organized by MT number |
| `urr` | Unresolved resonance probability tables, if available |
| `total_nu` | Total neutron yield per fission, when present |
| `fission_energy_release` | Fission energy release components, when present |

## 2. Inspect attributes

In [3]:
def get_attrs(filename):
    with h5py.File(DATA_ROOT / filename) as ff:
        nuclide = next(iter(ff.values()))
        return set(nuclide.attrs)

attrs = set()
for filename in df['path'].unique():
    attrs.update(get_attrs(filename))
attrs

{'A', 'Z', 'atomic_weight_ratio', 'metastable'}

## 3. Get metadata

In [4]:
def get_temperatures(filename):
    with h5py.File(DATA_ROOT / filename) as ff:
        nuclide = next(iter(ff.values()))

        temperatures = sorted(
            nuclide['kTs'],
            key=lambda t: int(t.removesuffix('K'))
        )

        return temperatures

paths = df['path'].drop_duplicates()
temperatures = paths.map(get_temperatures)

path2temperature = dict(zip(paths, temperatures))
df['temperatures'] = df['path'].map(path2temperature)
df

,materials,path,library,MT,temperatures
0,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,2,"[250K, 294K, 600K, 900K, 1200K, 2500K]"
1,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,102,"[250K, 294K, 600K, 900K, 1200K, 2500K]"
2,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,204,"[250K, 294K, 600K, 900K, 1200K, 2500K]"
3,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,301,"[250K, 294K, 600K, 900K, 1200K, 2500K]"
4,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,444,"[250K, 294K, 600K, 900K, 1200K, 2500K]"
...,...,...,...,...,...
219209,Fm257,jendl-5/neutron/Fm257.h5,jendl-5,750,"[250K, 294K, 600K, 900K, 1200K, 2500K]"
219210,Fm257,jendl-5/neutron/Fm257.h5,jendl-5,799,"[250K, 294K, 600K, 900K, 1200K, 2500K]"
219211,Fm257,jendl-5/neutron/Fm257.h5,jendl-5,800,"[250K, 294K, 600K, 900K, 1200K, 2500K]"
219212,Fm257,jendl-5/neutron/Fm257.h5,jendl-5,849,"[250K, 294K, 600K, 900K, 1200K, 2500K]"


All nuclear-data libraries in our collection provide cross sections at six temperatures:
250, 294, 600, 900, 1200, and 2500 K.

The exception is FENDL-3.2c, which provides data only at 294 K.

This is because the distributed FENDL-3.2c ACE files were processed at 293.6 K (rounded to 294 K by OpenMC). Additional temperatures would require Doppler broadening.

In [5]:
df['temperatures'].map(tuple).unique()

array([('250K', '294K', '600K', '900K', '1200K', '2500K'), ('294K',)],
      dtype=object)

In [6]:
combination = df['temperatures'].iloc[0]
df.loc[df['temperatures'].map(set) != set(combination), 'library'].unique()

<ArrowStringArray>
['fendl-3.2c']
Length: 1, dtype: str